# 自定义中间件-Wrap-style hooks
## 1、wrap_model_call的使用

### 1.1 基于装饰器的实现

In [1]:
  from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    profile={"max_input_tokens":128_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [2]:

from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
    request.messages[-1].content += "---> wrap_model_call_before <---"

    # 模型的调用
    response = handler(request)

    response.result[0].content += "---> wrap_model_call_after <---"

    return response

In [3]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！很高兴见到你！😊

我是DeepSeek，由深度求索公司创造的AI助手。我可以帮你解答问题、进行创作、分析数据、提供建议等等。

有什么我可以帮你的吗？无论是学习、工作还是生活上的问题，都可以随时向我提问！我会尽我所能为你提供帮助。

顺便说一句，我目前是完全免费的，支持1M超长上下文（可以一次性处理像《三体》三部曲那样的长文本），还支持文件上传和联网搜索功能（需要手动开启哦）。

那么，今天想聊点什么呢？🤗---> wrap_model_call_after <---


### 1.2 基于类的实现

In [4]:
from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call_before <---"

        # 模型的调用
        response = handler(request)

        response.result[0].content += "---> wrap_model_call_after <---"

        return response


agent = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！看起来你的消息被一个占位符截断了，可能是代码或提示词的一部分。

请问有什么我可以帮你的吗？如果你是想测试某个功能或讨论某个话题，请直接告诉我，我会尽力协助！😊---> wrap_model_call_after <---


## 2、wrap_tool_call的使用

### 2.1 基于装饰器的实现

In [5]:

from langchain_core.tools import tool
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call


@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                              ) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用结果：{result}")
    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '北京', 'is_forcast': False}
原始参数调用结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_00_4coMPH7ccnzon34F8Ue35829'
更新以后的参数：{'city': '北京', 'is_forcast': True}
更新以后的参数调用结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_4coMPH7ccnzon34F8Ue35829'
================================ Human Message =================================

帮我查询北京今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_4coMPH7ccnzon34F8Ue35829)
 Call ID: call_00_4coMPH7ccnzon34F8Ue35829
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================== Ai Message ==================================

根据查询结果，北京今天的天气**不错**，明天的天气**也很好**！☀️

需要我帮你查询其他城市的天气吗？


### 2.2 基于类的实现

In [7]:

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self, request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                       ) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{result}")

        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)

        print(f"更新以后的参数：{request.tool_call['args']}")
        print(f"更新以后的参数调用结果：{result}")
        return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[WrapToolCallMiddleware()]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询上海今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '上海', 'is_forcast': False}
原始参数调用结果：content='上海今天天气不错' name='get_weather' tool_call_id='call_00_v2M6DRY060NXRgH5AaSl6771'
更新以后的参数：{'city': '上海', 'is_forcast': True}
更新以后的参数调用结果：content='上海今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_v2M6DRY060NXRgH5AaSl6771'
================================ Human Message =================================

帮我查询上海今天的天气如何？
================================== Ai Message ==================================

我来帮您查询上海今天的天气情况。
Tool Calls:
  get_weather (call_00_v2M6DRY060NXRgH5AaSl6771)
 Call ID: call_00_v2M6DRY060NXRgH5AaSl6771
  Args:
    city: 上海
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

上海今天天气不错
明天天气也很好
================================== Ai Message ==================================

根据查询结果，为您汇报上海的天气情况：

**今天（上海）**：天气不错 ☀️

另外我还为您查看了明天的天气预报，**明天天气也很好**。

如果您需要更详细的天气信息，比如温度、风力等具体数据，或者想查询其他城市的天气，随时告诉我！
